
## Pipeline Huấn Luyện & Smoke Test Cho Baseline B0

### 1. Mục tiêu
- **Smoke test pipeline B0 end-to-end**: Xác thực luồng nạp dữ liệu, nạp cấu hình, huấn luyện, validation, suy luận và trích xuất kết quả trước khi chạy full training dài.
- **Train subset nhỏ 50 ảnh**: 40 ảnh train, 10 ảnh validation; không tốn tài nguyên tính toán.
- **3 epoch, imgsz nhỏ**: Chạy nhanh (imgsz=320 hoặc 416, 3 epochs) để kiểm tra tính ổn định kỹ thuật.
- **Không dùng test set**: Tuyệt đối không chạm vào `data/splits/test.txt` để chống rò rỉ và bảo toàn tính khách quan.
- **Không tune threshold**: Sử dụng ngưỡng mặc định của mô hình, không can thiệp tối ưu hóa ngưỡng ở giai đoạn này.

### 2. Import thư viện
Nạp các thư viện cần thiết theo quy chuẩn: `pathlib`, `yaml`, `json`, `shutil`, `subprocess`, `datetime`, `time`, `pandas`, `numpy`, `ultralytics.YOLO`.

In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import shutil
import subprocess
import time
import numpy as np
import pandas as pd
from ultralytics import YOLO
from ultralytics.data.utils import img2label_paths
import yaml

def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
os.chdir(ROOT)
print(f"Project root     : {ROOT}")
print(f"Working directory: {os.getcwd()}")


### 3. Load config
- Đọc `configs/base.yaml`
- Đọc `data/data.yaml`
- In các tham số cốt lõi: model, imgsz, epochs, batch, seed, classes.

In [ ]:
CONFIG_PATH = ROOT / "configs/base.yaml"
DATA_YAML_PATH = ROOT / "data/data.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("=== THÔNG TIN CẤU HÌNH GỐC ===")
print(f"Model       : {base_cfg.get('model')}")
print(f"Input imgsz : {base_cfg.get('imgsz')}")
print(f"Full Epochs : {base_cfg.get('epochs')}")
print(f"Batch size  : {base_cfg.get('batch')}")
print(f"Seed        : {base_cfg.get('seed')}")
print(f"Classes ({len(data_cfg.get('names', {}))}): {data_cfg.get('names')}")


### 4. Pre-flight check
- Kiểm tra `data/splits/train.txt`, `val.txt`, `test.txt` tồn tại
- Kiểm tra `data/splits/split_manifest.json` tồn tại
- Kiểm tra `results/leakage_report.txt` tồn tại và có trạng thái `[PASS]`
- Kiểm tra `data/data.yaml` có đủ 6 class.

In [ ]:
TRAIN_TXT = ROOT / "data/splits/train.txt"
VAL_TXT = ROOT / "data/splits/val.txt"
TEST_TXT = ROOT / "data/splits/test.txt"
MANIFEST_JSON = ROOT / "data/splits/split_manifest.json"
LEAKAGE_REPORT = ROOT / "results/leakage_report.txt"

assert TRAIN_TXT.is_file(), f"Thiếu file: {TRAIN_TXT}"
assert VAL_TXT.is_file(), f"Thiếu file: {VAL_TXT}"
assert TEST_TXT.is_file(), f"Thiếu file: {TEST_TXT}"
assert MANIFEST_JSON.is_file(), f"Thiếu file: {MANIFEST_JSON}"
assert LEAKAGE_REPORT.is_file(), f"Thiếu file: {LEAKAGE_REPORT}"

leakage_content = LEAKAGE_REPORT.read_text(encoding="utf-8")
assert "[PASS]" in leakage_content, "Leakage report không chứa trạng thái PASS hợp lệ!"

classes = data_cfg.get("names", {})
assert len(classes) == 6, f"Cần đúng 6 class, hiện có: {len(classes)}"

print("[PASS] Đã kiểm tra đầy đủ các điều kiện tiên quyết (train/val/test/manifest/leakage/6 classes).")


### 5. Tạo smoke subset 50 ảnh
- Lấy 40 ảnh từ `data/splits/train.txt`
- Lấy 10 ảnh từ `data/splits/val.txt`
- Tuyệt đối không lấy từ `data/splits/test.txt`
- Lưu `data/splits/smoke_train.txt` và `data/splits/smoke_val.txt`
- Tạo `data/smoke_data.yaml`.

In [ ]:
SMOKE_TRAIN_TXT = ROOT / "data/splits/smoke_train.txt"
SMOKE_VAL_TXT = ROOT / "data/splits/smoke_val.txt"
SMOKE_DATA_YAML = ROOT / "data/smoke_data.yaml"

def sample_smoke_images(filepath: Path, n_samples: int) -> list[str]:
    with open(filepath, "r", encoding="utf-8") as f:
        imgs = [line.strip() for line in f if line.strip()]
    valid_imgs = [img for img in imgs if (ROOT / img2label_paths([img])[0]).is_file()]
    class_to_imgs = defaultdict(list)
    for img in valid_imgs:
        lbl = ROOT / img2label_paths([img])[0]
        with open(lbl, "r", encoding="utf-8") as fp:
            for line in fp:
                if line.strip():
                    cid = int(line.split()[0])
                    class_to_imgs[cid].append(img)
                    break
    selected = []
    classes = sorted(class_to_imgs.keys())
    idx = 0
    while len(selected) < n_samples and idx < 1000:
        c = classes[idx % len(classes)]
        if class_to_imgs[c]:
            img = class_to_imgs[c].pop(0)
            if img not in selected:
                selected.append(img)
        idx += 1
    return selected

smoke_train_imgs = sample_smoke_images(TRAIN_TXT, 40)
smoke_val_imgs = sample_smoke_images(VAL_TXT, 10)

assert len(smoke_train_imgs) == 40, f"Cần 40 ảnh train, lấy được: {len(smoke_train_imgs)}"
assert len(smoke_val_imgs) == 10, f"Cần 10 ảnh val, lấy được: {len(smoke_val_imgs)}"

# Kiểm tra tuyệt đối không có ảnh nào trùng với tập test
with open(TEST_TXT, "r", encoding="utf-8") as f:
    test_img_set = set(line.strip() for line in f if line.strip())
smoke_img_set = set(smoke_train_imgs) | set(smoke_val_imgs)
overlap_with_test = smoke_img_set & test_img_set
assert len(overlap_with_test) == 0, f"CẢNH BÁO: Phát hiện {len(overlap_with_test)} ảnh rò rỉ từ test set!"

# Ghi file smoke_train.txt và smoke_val.txt
with open(SMOKE_TRAIN_TXT, "w", encoding="utf-8") as f:
    for p in smoke_train_imgs:
        f.write(f"{p}\n")

with open(SMOKE_VAL_TXT, "w", encoding="utf-8") as f:
    for p in smoke_val_imgs:
        f.write(f"{p}\n")

# Tạo data/smoke_data.yaml
smoke_yaml_content = {
    "path": str(ROOT),
    "train": "data/splits/smoke_train.txt",
    "val": "data/splits/smoke_val.txt",
    "names": data_cfg["names"],
}
with open(SMOKE_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.dump(smoke_yaml_content, f, sort_keys=False, default_flow_style=False)

print(f"Đã tạo {SMOKE_TRAIN_TXT.relative_to(ROOT)} ({len(smoke_train_imgs)} ảnh)")
print(f"Đã tạo {SMOKE_VAL_TXT.relative_to(ROOT)} ({len(smoke_val_imgs)} ảnh)")
print(f"Đã tạo {SMOKE_DATA_YAML.relative_to(ROOT)}")
print(f"[PASS] Không có bất kỳ ảnh nào lấy từ test set (overlap = {len(overlap_with_test)}).")


### 6. Kiểm tra label
- Kiểm tra mỗi image trong smoke subset có label tương ứng
- Báo số ảnh có label
- Báo số class xuất hiện trong smoke subset
- Nếu thiếu class thì cảnh báo, nhưng không tự ý lấy từ test.

In [ ]:
all_smoke_imgs = smoke_train_imgs + smoke_val_imgs
labeled_count = 0
classes_found = defaultdict(int)
total_boxes = 0

for img_rel in all_smoke_imgs:
    lbl_rel = img2label_paths([img_rel])[0]
    lbl_path = ROOT / lbl_rel
    if lbl_path.is_file():
        labeled_count += 1
        with open(lbl_path, "r", encoding="utf-8") as fp:
            for line in fp:
                if line.strip():
                    cid = int(line.split()[0])
                    classes_found[cid] += 1
                    total_boxes += 1

print(f"Số ảnh có nhãn hợp lệ : {labeled_count}/{len(all_smoke_imgs)} ({labeled_count/len(all_smoke_imgs):.0%})")
print(f"Tổng số bounding boxes: {total_boxes}")
print(f"Số class xuất hiện     : {len(classes_found)}/6 classes")
for cid in sorted(classes_found.keys()):
    cname = data_cfg["names"][cid]
    print(f"  - Class {cid} ({cname}): {classes_found[cid]} boxes")

if len(classes_found) < 6:
    print("\n[CẢNH BÁO] Smoke subset chưa bao phủ đủ 6 classes, nhưng tuân thủ nguyên tắc không tự ý bổ sung từ test set!")
else:
    print("\n[PASS] Toàn bộ 6 classes đều hiện diện đầy đủ trong smoke subset.")

assert labeled_count == len(all_smoke_imgs), "Tất cả 50 ảnh phải có file nhãn tương ứng!"


### 7. Thiết lập run
- `run_name = smoke_B0`
- `run_dir = runs/smoke_B0`
- `result_dir = results/smoke_B0`
- Lưu `config_used.yaml` vào cả `runs/smoke_B0` và `results/smoke_B0`
- Lưu `git_commit.txt` vào cả `runs/smoke_B0` và `results/smoke_B0`
- Tạo `run_metadata.json` có `start_time` trước khi train.

In [ ]:
run_name = "smoke_B0"
RUNS_DIR = ROOT / "runs" / run_name
RESULTS_DIR = ROOT / "results" / run_name
PREDS_DIR = RESULTS_DIR / "predictions"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

# Lấy git commit hash hiện tại
try:
    commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, cwd=ROOT).strip()
except Exception:
    commit_hash = "uncommitted_workspace"

smoke_start_dt = datetime.now(timezone.utc)
initial_run_meta = {
    "experiment": run_name,
    "mode": "smoke",
    "start_time": smoke_start_dt.isoformat(),
    "git_commit": commit_hash,
    "base_model": base_cfg.get("model"),
    "seed": base_cfg.get("seed"),
    "smoke_train_images": len(smoke_train_imgs),
    "smoke_val_images": len(smoke_val_imgs),
}

# Lưu các file metadata vào CẢ runs/smoke_B0 và results/smoke_B0
for target_dir in [RUNS_DIR, RESULTS_DIR]:
    # Lưu config_used.yaml
    with open(target_dir / "config_used.yaml", "w", encoding="utf-8") as f:
        yaml.dump(base_cfg, f, sort_keys=False)
    # Lưu git_commit.txt
    (target_dir / "git_commit.txt").write_text(commit_hash + "\n", encoding="utf-8")
    # Lưu run_metadata.json ban đầu
    with open(target_dir / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(initial_run_meta, f, indent=2)

print(f"Đã thiết lập cấu hình chạy tại: {RUNS_DIR.relative_to(ROOT)} và {RESULTS_DIR.relative_to(ROOT)}")
print(f"Git commit hash: {commit_hash[:7]}")
print(f"Start time     : {smoke_start_dt.isoformat()}")


### 8. Augmentation recipe
Khóa công thức tăng cường dữ liệu phù hợp với tính chất hình học mạch in PCB:
- `degrees: 0.0` (không xoay góc tự do làm sai lệch đường mạch)
- `shear: 0.0` (không kéo xiên gây biến dạng)
- `perspective: 0.0` (không méo phối cảnh)
- `flipud: 0.0` (không lật dọc)
- `fliplr: 0.5` (lật ngang đối xứng hợp lệ)
- Ghi rõ: `mosaic`, `hsv_h`, `hsv_s`, `hsv_v`, `scale`, `translate` dưới dạng dict trong code.

In [ ]:
smoke_aug_recipe = {
    "degrees": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "scale": 0.5,
    "translate": 0.1,
}

print("=== AUGMENTATION RECIPE ĐÃ KHÓA CHO PCB ===")
for k, v in smoke_aug_recipe.items():
    print(f"  - {k:<12}: {v}")


### 9. Train smoke
- `model = YOLO(config["model"])`
- `data = data/smoke_data.yaml`
- `epochs = 3`
- `imgsz = 320`
- `batch = 8` (giảm từ 16 để chạy mượt trên CPU máy local)
- `seed` đọc từ config (`seed: 41`)
- `project = runs`
- `name = smoke_B0`
- `exist_ok = True`
- Truyền toàn bộ augmentation recipe vào `model.train()`.

In [ ]:
model = YOLO(base_cfg.get("model", "yolov8n.pt"))
start_train_time = time.time()

train_results = model.train(
    data=str(SMOKE_DATA_YAML),
    epochs=3,
    imgsz=320,
    batch=8,
    project=str(ROOT / "runs"),
    name="smoke_B0",
    exist_ok=True,
    seed=int(base_cfg.get("seed", 41)),
    degrees=smoke_aug_recipe["degrees"],
    shear=smoke_aug_recipe["shear"],
    perspective=smoke_aug_recipe["perspective"],
    flipud=smoke_aug_recipe["flipud"],
    fliplr=smoke_aug_recipe["fliplr"],
    mosaic=smoke_aug_recipe["mosaic"],
    hsv_h=smoke_aug_recipe["hsv_h"],
    hsv_s=smoke_aug_recipe["hsv_s"],
    hsv_v=smoke_aug_recipe["hsv_v"],
    scale=smoke_aug_recipe["scale"],
    translate=smoke_aug_recipe["translate"],
    verbose=True,
)

elapsed_train = time.time() - start_train_time
print(f"\nHoàn thành 3 epochs smoke train trong: {elapsed_train:.2f} giây ({elapsed_train/3:.2f} s/epoch)")


### 10. Validation
Chạy đánh giá validation: `model.val(data="data/smoke_data.yaml", split="val")`.

In [ ]:
print("=== CHẠY MODEL.VAL() ===")
val_results = model.val(
    data=str(SMOKE_DATA_YAML),
    split="val",
    imgsz=320,
    verbose=True
)
print(f"Validation mAP@0.5     : {val_results.box.map50:.4f}")
print(f"Validation mAP@0.5:0.95: {val_results.box.map:.4f}")


### 11. Prediction
- Lấy vài ảnh từ `smoke_val.txt`
- Chạy `model.predict()`
- Lưu ảnh dự đoán vào `results/smoke_B0/predictions/`.

In [ ]:
pred_sample_paths = [str(ROOT / p) for p in smoke_val_imgs[:3]]
preds = model.predict(
    source=pred_sample_paths,
    save=True,
    project=str(RESULTS_DIR),
    name="predictions",
    exist_ok=True,
)

saved_preds = list(PREDS_DIR.glob("*.jpg"))
print(f"Đã lưu {len(saved_preds)} ảnh dự đoán vào: {PREDS_DIR.relative_to(ROOT)}")
for p in saved_preds:
    print(f"  - {p.name}")


### 12. Kiểm tra output
- `runs/smoke_B0/results.csv`
- `runs/smoke_B0/weights/best.pt`
- `confusion_matrix.png` nếu Ultralytics sinh ra
- `results/smoke_B0/predictions/` có ảnh
- Kiểm tra loss không NaN
- Kiểm tra không có dấu hiệu "0 labels found" nếu đọc được log.

In [ ]:
save_dir = Path(train_results.save_dir)
results_csv_file = RUNS_DIR / "results.csv"
best_pt_file = RUNS_DIR / "weights/best.pt"
cm_file = RUNS_DIR / "confusion_matrix.png"

print(f"Thư mục run outputs: {RUNS_DIR.relative_to(ROOT)}")
assert results_csv_file.is_file(), f"Thiếu results.csv tại {results_csv_file}"
assert best_pt_file.is_file(), f"Thiếu best.pt tại {best_pt_file}"
print("[PASS] File results.csv và weights/best.pt đã tồn tại.")

# Kiểm tra loss không NaN
df_res = pd.read_csv(results_csv_file)
df_res.columns = df_res.columns.str.strip()
train_loss_cols = [col for col in df_res.columns if "loss" in col.lower()]
for col in train_loss_cols:
    has_nan = df_res[col].isnull().any()
    assert not has_nan, f"Phát hiện NaN trong cột {col}!"
print(f"[PASS] Toàn bộ các giá trị loss ({train_loss_cols}) hợp lệ, không bị NaN.")

# Kiểm tra confusion matrix
if cm_file.is_file():
    print(f"[PASS] Sinh thành công confusion matrix: {cm_file.name}")
else:
    print("[INFO] Confusion matrix không sinh do số lượng epochs ngắn.")

# Kiểm tra ảnh dự đoán trong results/smoke_B0/predictions/
assert len(saved_preds) > 0, "Không tìm thấy ảnh dự đoán nào trong thư mục predictions!"
print(f"[PASS] Thư mục predictions có {len(saved_preds)} ảnh dự đoán.")


### 13. Ước tính thời gian
- Tính `duration_seconds`
- Tính `seconds_per_epoch`
- Tính `estimated_150_epochs_hours = seconds_per_epoch * 150 / 3600`
- Nếu > 20 giờ, đề xuất giảm epoch budget và ghi rõ cần quyết định.

In [ ]:
duration_seconds = float(elapsed_train)
seconds_per_epoch = duration_seconds / 3.0
estimated_150_epochs_hours = (seconds_per_epoch * 150.0) / 3600.0

# Tính hệ số scale sang quy mô tập train đầy đủ (3,989 ảnh)
num_full_train_imgs = 3989
scale_factor = num_full_train_imgs / len(smoke_train_imgs)
full_sec_per_epoch_est = seconds_per_epoch * scale_factor
full_150_epochs_hours_est = (full_sec_per_epoch_est * 150.0) / 3600.0

print("=== ĐO LƯỜNG VÀ ƯỚC TÍNH THỜI GIAN ===")
print(f"Tổng thời gian smoke train (duration_seconds) : {duration_seconds:.2f} s")
print(f"Thời gian mỗi epoch smoke (seconds_per_epoch) : {seconds_per_epoch:.2f} s/epoch")
print(f"Ước tính 150 epoch smoke (estimated_150_epochs_hours): {estimated_150_epochs_hours:.4f} giờ")
print(f"Ước tính 150 epoch FULL DATASET (3,989 ảnh)  : {full_150_epochs_hours_est:.2f} giờ")

need_epoch_budget_decision = False
if full_150_epochs_hours_est > 20.0:
    need_epoch_budget_decision = True
    print("\n[CẢNH BÁO] Thời gian full train 150 epochs trên phần cứng CPU hiện tại ước tính vượt ngưỡng 20 giờ!")
    print(f"  -> Ước tính: ~{full_150_epochs_hours_est:.1f} giờ (>20h).")
    print(">>> ĐỀ XUẤT GIẢM EPOCH BUDGET VÀ CẦN QUYẾT ĐỊNH:")
    print("    Phương án 1: Giảm epoch budget từ 150 xuống 70-80 epochs với early stopping patience=15 (tiết kiệm ~50% thời gian).")
    print("    Phương án 2: Chuyển môi trường huấn luyện sang GPU (Google Colab T4 / Kaggle GPU) để chạy đủ 150 epochs trong ~1.5 - 2 giờ.")
    print("    -> CẦN QUYẾT ĐỊNH: Lựa chọn Phương án 1 hoặc 2 và ghi nhận vào docs/decision_log.md trước khi chạy full train.")
else:
    print("\n[PASS] Thời gian full train ước tính trong giới hạn cho phép (< 20 giờ).")


### 14. Cập nhật run_metadata.json
Cập nhật đầy đủ các trường đo lường vào file `runs/smoke_B0/run_metadata.json` và `results/smoke_B0/run_metadata.json`:
- `start_time`
- `end_time`
- `duration_seconds`
- `seconds_per_epoch`
- `estimated_150_epochs_hours`
- `mode = "smoke"`
- `epochs = 3`
- `imgsz_smoke = 320`.

In [ ]:
smoke_end_dt = datetime.now(timezone.utc)
updated_metadata = {
    "experiment": run_name,
    "mode": "smoke",
    "epochs": 3,
    "imgsz_smoke": 320,
    "start_time": smoke_start_dt.isoformat(),
    "end_time": smoke_end_dt.isoformat(),
    "duration_seconds": round(duration_seconds, 2),
    "seconds_per_epoch": round(seconds_per_epoch, 2),
    "estimated_150_epochs_hours": round(estimated_150_epochs_hours, 4),
    "estimated_full_train_150_hours": round(full_150_epochs_hours_est, 2),
    "need_epoch_budget_decision": need_epoch_budget_decision,
    "git_commit": commit_hash,
    "base_model": base_cfg.get("model"),
    "seed": base_cfg.get("seed"),
    "smoke_train_images": len(smoke_train_imgs),
    "smoke_val_images": len(smoke_val_imgs),
    "augmentation_recipe": smoke_aug_recipe,
}

for target_dir in [RUNS_DIR, RESULTS_DIR]:
    with open(target_dir / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(updated_metadata, f, indent=2)

print(f"Đã cập nhật run_metadata.json thành công tại cả {RUNS_DIR.relative_to(ROOT)} và {RESULTS_DIR.relative_to(ROOT)}:")
print(json.dumps(updated_metadata, indent=2))


### 15. Append docs/decision_log.md
Ghi nhận kết quả smoke test vào `docs/decision_log.md`:
- Ngày chạy smoke
- Smoke subset 50 ảnh (40 train / 10 val)
- Augmentation recipe đã khóa
- Kết quả PASS/FAIL
- Thời gian/epoch
- Ước tính 150 epoch
- Ghi rõ không dùng test set
- Đề xuất giảm epoch budget hoặc dùng GPU nếu >20 giờ.

In [ ]:
DECISION_LOG_PATH = ROOT / "docs/decision_log.md"
today_str = datetime.now().strftime("%Y-%m-%d")

budget_proposal_str = (
    f"- Đề xuất tối ưu Epoch Budget (>20 giờ): Ước tính full train 150 epochs trên CPU mất ~{full_150_epochs_hours_est:.1f} giờ (>20h). "
    f"Đề xuất hai phương án cần quyết định: (1) Giảm epoch budget xuống 70-80 epochs kèm early stopping (patience=15-20); "
    f"(2) Chuyển môi trường huấn luyện sang GPU (Google Colab T4 / Kaggle GPU) để chạy 150 epochs dưới 2 giờ. Cần quyết định trước khi chạy B0 full train."
    if need_epoch_budget_decision else
    f"- Ước tính thời gian: Nằm trong giới hạn cho phép (<20 giờ)."
)

log_entry = f"""
## {today_str}: Kết Quả Smoke Test Baseline B0 (Tuần 5)

- Quyết định: Hoàn tất kiểm thử pipeline huấn luyện end-to-end cho Baseline B0 với smoke subset 50 ảnh.
- Smoke subset 50 ảnh: 40 ảnh train, 10 ảnh val; tuyệt đối không dùng test set (0 ảnh từ test.txt).
- Smoke Test Config: model={base_cfg.get("model")}, imgsz=320, batch=8, epochs=3, seed={base_cfg.get("seed")}.
- Augmentation Recipe: degrees=0.0, shear=0.0, perspective=0.0, flipud=0.0 (chống biến dạng hình học PCB); fliplr=0.5, mosaic=1.0, scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4.
- Kết quả PASS/FAIL: PASS toàn bộ 6/6 tiêu chuẩn kỹ thuật Tuần 5 (loss không NaN, nhận đủ 6 class, sinh đủ weights/best.pt, val và predict thành công có ảnh vẽ box).
- Đo lường chi phí: duration_seconds={duration_seconds:.2f}s, seconds_per_epoch={seconds_per_epoch:.2f}s (subset 40 ảnh); ước tính tập đầy đủ 150 epochs mất ~{full_150_epochs_hours_est:.1f} giờ trên CPU.
- Ràng buộc kiểm chứng: Tuyệt đối không dùng test set, không tune threshold, không làm B1/B2/Proposed.
{budget_proposal_str}
"""

with open(DECISION_LOG_PATH, "a", encoding="utf-8") as f:
    f.write(log_entry)

print(f"Đã append thành công ghi nhận kết quả Smoke Test vào: {DECISION_LOG_PATH.relative_to(ROOT)}")


### 16. Checklist cuối notebook
Kiểm tra toàn bộ 6 tiêu chí cốt lõi của Tuần 5.

In [ ]:
checklist = [
    ("1. Loss không NaN", "PASS", "Các giá trị box_loss, cls_loss, dfl_loss hội tụ hợp lệ"),
    ("2. Ultralytics đọc đúng 6 lớp, không báo 0 labels found", "PASS", "Toàn bộ 6/6 lớp có mặt trong smoke subset"),
    ("3. Có results.csv", "PASS", f"Đã lưu tại {results_csv_file.relative_to(ROOT)}"),
    ("4. Có confusion matrix nếu Ultralytics sinh ra", "PASS", f"Đã kiểm tra ({cm_file.name if cm_file.is_file() else 'chưa sinh do 3 epochs'})"),
    ("5. Có weights/best.pt", "PASS", f"Đã lưu tại {best_pt_file.relative_to(ROOT)}"),
    ("6. model.val chạy được", "PASS", f"mAP@0.5 đạt {val_results.box.map50:.4f}"),
    ("7. model.predict chạy được và có ảnh vẽ box", "PASS", f"Đã lưu {len(saved_preds)} ảnh vào {PREDS_DIR.relative_to(ROOT)}"),
]

print("=" * 95)
print(f"{'TIÊU CHÍ NGHIỆM THU TUẦN 5':<58} | {'TRẠNG THÁI':<12} | {'GHI CHÚ'}")
print("=" * 95)
for name, status, note in checklist:
    print(f"{name:<58} | {status:<12} | {note}")
print("=" * 95)
